last modified date : 2026.05  
제작 : 모두의연구소

# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가

# 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 직접 구현해 보았습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 데이터셋 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이번 실습이 끝나면 다음을 직접 말할 수 있게 됩니다.
- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 ‘얼마나 좋아졌는지’를 **숫자로** 보여주는 방법

## Step 0 : 설치와 준비  
Day 1과 동일하게 Colab에서 진행한다고 가정합니다.

In [1]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: ragas 0.2.10
Uninstalling ragas-0.2.10:
  Successfully uninstalled ragas-0.2.10
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-openai 0.1.25
Uninstalling langchain-openai-0.1.25:
  Successfully uninstalled langchain-openai-0.1.25
Found existing installation: langchain-text-splitters 0.2.4
Uninstalling langchain-text-splitters-0.2.4:
  Successfully uninstalled langchain-text-splitters-0.2.4
Found existing installation: langchain-chroma 0.1.4
Uninstalling langchain-chroma-0.1.4:
  Successfully uninstalled langchain-chroma-0.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

> ⚠️ **위 설치 셀(Step 0)을 실행한 뒤 반드시 [런타임 > 세션 다시 시작 (Restart session)]을 한 번 눌러주세요.**
>
> 이 셀은 Colab에 기본 설치된 langchain을 제거하고 `0.2.x` / `ragas 0.2.10` 조합으로 다운그레이드합니다. 이미 메모리에 로드된 패키지를 교체하는 것이라 Colab이 재시작을 요구합니다.
>
> 재시작 후에는 **설치 셀은 다시 실행하지 말고** 이 셀 아래(키 설정)부터 순서대로 실행하면 됩니다.

In [2]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()  # RAGAS가 Colab의 비동기 이벤트 루프와 충돌하지 않도록

In [3]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [4]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [6]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 중에 이루어진 주요 개선 프로젝트는 어떤 것들이 있나요?']


검색된 문서 수: 6
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [7]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """
    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    # 각 쿼리의 결과 리스트를 순회하면서 문서마다 RRF 점수를 누적
    for docs in results_per_query:
        for rank, doc in enumerate(docs):  # rank 는 0부터
            key = doc.page_content
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    # scores 값이 큰 순서로 정렬해서 상위 top_k 개의 Document 를 반환
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[key] for key, _ in ranked[:top_k]]


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")


확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 재직 시절 2004년에 개선한 것은 무엇인지?

RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [8]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 계획'을 통해 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용 차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 이동 편의성을 크게 향상시키고, 대기 오염 문제 해결에도 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [9]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [10]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [11]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "당신은 사용자 질문을 분석해 외부 문서 검색이 필요한지 판단하는 라우터입니다.\n"
    "질문이 특정 사실, 인물, 사건, 수치 등 외부 지식(문서)을 참고해야 정확히 답할 수 있다면 YES,\n"
    "일반 상식, 단순 계산, 정의, 인사말처럼 LLM 자체 지식만으로 충분히 답할 수 있다면 NO 라고 답하세요.\n"
    "오직 한 단어(YES 또는 NO)만 출력하세요. 다른 설명은 붙이지 마세요.\n\n"
    "질문: {question}"
)

# (2) 답변 자가 비평 프롬프트
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "다음 [문서] 내용과 [답변]을 비교해, 답변의 핵심 주장이 문서 내용으로 충분히 뒷받침되는지 판단하세요.\n"
    "문서에 없는 내용을 답변이 지어냈거나 문서와 답변이 무관하다면 NOT_SUPPORTED,\n"
    "답변이 문서 내용에 근거하고 있다면 SUPPORTED 라고 답하세요.\n"
    "오직 한 단어(SUPPORTED 또는 NOT_SUPPORTED)만 출력하세요. 다른 설명은 붙이지 마세요.\n\n"
    "[문서]\n{context}\n\n[답변]\n{answer}"
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)


[1] Retrieve 필요? -> YES
[3] 시도 1 — 자가 비평: NOT_SUPPORTED
[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색
[3] 시도 2 — 자가 비평: NOT_SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [12]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [13]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [14]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [15]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.575         0.850
answer_relevancy       0.291         0.262
context_precision      0.692         0.850
context_recall         0.750         0.850

Delta (Advanced - Naive):
faithfulness         0.275
answer_relevancy    -0.029
context_precision    0.158
context_recall       0.100
dtype: float64


### 결과 해석 가이드

위 비교표를 처음 보면 **‘Advanced 가 더 나쁜 거 아닌가?’** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 결과 해석 방법을 정리합니다.

**1. `context_precision` 의 개선 (+) 이 Advanced RAG 의 핵심 효과**
검색 결과의 ‘상단’에 정답 문단을 두는 일을 Reranker 가 잘 했다는 의미. Δ가 0.05~0.15 정도면 잘 작동.

**2. `context_recall = 1.0` 으로 포화될 수 있다**
unique context 가 800개 정도면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 이 지표는 더 큰 DB(수만 문서)에서 차이가 드러납니다.

**3. `faithfulness` 가 살짝 떨어질 수 있다**
Reranker 가 컨텍스트를 ‘짧고 집중’ 시키면 LLM이 그 좁은 정보에서 답을 만들 때 일부 주장이 “미뒷받침” 으로 채점되어 점수가 약간 내려갈 수 있음. **정상 범위 (-0.1 이내)**.

**4. `answer_relevancy` 가 0.2~0.4 로 낮은 이유 — KorQuAD 의 구조적 특성**
KorQuAD 정답은 *‘대중교통체계’* 같이 한 단어~한 구절. RAG 답변도 짧게 나오는데, RAGAS 의 `answer_relevancy` 는 **답변에서 질문을 역추론**해 원래 질문과의 유사도를 계산합니다. 답변이 한 단어면 역추론이 흐려져 점수가 낮아집니다. **모델 잘못이 아닌 데이터셋 특성**.

**5. 표본 20개로도 Δ가 ±0.05 이내면 ‘차이 없음’으로 봐야 한다**
20문항에서 ±0.05 는 표본 noise. 더 확실한 판단이 필요하면 `scipy.stats.ttest_rel` 로 통계 검정을 하거나 50~100문항으로 늘려야 합니다.

**6. 한국어 짧은 정답 벤치마크의 한계**
KorQuAD/KLUE-MRC 처럼 정답이 짧은 extractive QA 벤치마크는 `context_precision` 위주로 평가 효과를 봐야 하고, `answer_relevancy` 는 절대값보다 **Naive 대비 상대 변화**로 읽어야 합니다.

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer**:  
이번 실행에서 Advanced RAG(Reranker)가 가장 크게 기여한 지표는 faithfulness이며, context_precision·context_recall도 동반 상승해 검색 품질 개선이 답변 신뢰도로 이어졌음을 보여줍니다. 다만 20개 표본이라 표본 변동성은 감안해서 해석해야 합니다.

## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [16]:
from datasets import load_dataset

ds_klue = load_dataset("klue/klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [17]:
import random
from langchain_core.documents import Document

# is_impossible=True (답할 수 없는 질문) 제외
ds_klue_answerable = ds_klue.filter(lambda x: not x["is_impossible"])

# 300개 샘플링 (재사용을 위해 list 로 고정)
klue_samples = list(ds_klue_answerable.shuffle(seed=42).select(range(300)))

# context 기준 중복 제거
unique_klue = {}
for ex in klue_samples:
    if ex["context"] not in unique_klue:
        unique_klue[ex["context"]] = ex["title"]

context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique_klue.items()]

print(f"unique context 개수: {len(context_docs)}")
print(context_docs[0].page_content[:200])


unique context 개수: 299
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [18]:
db_klue = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(context_docs), BATCH):
    db_klue.add_documents(context_docs[i:i+BATCH])

print(f"db_klue 적재 완료 — {len(context_docs)}개 문서")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


db_klue 적재 완료 — 299개 문서


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [19]:
questions_klue = [ex["question"] for ex in klue_samples[:20]]
ground_truths_klue = [ex["answers"]["text"][0] for ex in klue_samples[:20]]

print(f"questions_klue: {len(questions_klue)}개")
print("샘플 질문:", questions_klue[0])
print("샘플 정답:", ground_truths_klue[0])


questions_klue: 20개
샘플 질문: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
샘플 정답: 두 개


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [20]:
naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})

naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("Q:", questions_klue[0])
print("A:", naive_chain_klue.invoke(questions_klue[0]))


Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
A: 200여 개의 계좌입니다.


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [21]:
multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

docs_mq_klue = multi_query_retriever_klue.invoke(questions_klue[0])
print(f"검색된 문서 수: {len(docs_mq_klue)}")
print("---")
print(docs_mq_klue[0].page_content[:300])


INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹 피해를 입은 리플 통장의 총 개수는 어떻게 되나요?']


검색된 문서 수: 6
---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [22]:
HYDE_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "당신은 뉴스 기자입니다. 다음 질문에 대해 기사 본문처럼 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator_klue = HYDE_PROMPT_KLUE | llm | StrOutputParser()

def hyde_retrieve_klue(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 db_klue 에서 검색"""
    hypothetical = hyde_generator_klue.invoke({"question": question})
    return db_klue.similarity_search(hypothetical, k=k), hypothetical

docs_hyde_klue, hyp_klue = hyde_retrieve_klue(questions_klue[0])
print("가상 답변(HyDE):\n", hyp_klue[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde_klue))
print("첫 문서:", docs_hyde_klue[0].page_content[:200])


가상 답변(HyDE):
 최근 국내에서 발생한 해킹 사건으로 인해 리플(XRP)이 포함된 통장 계좌 수가 급증한 것으로 나타났습니다. 금융감독원에 따르면, 이번 해킹으로 피해를 입은 계좌는 약 1,200개로 추정되며, 이들 계좌에서 리플을 포함한 암호화폐가 무단으로 인출된 사례가 다수 보고되었습니다. 전문가들은 해킹 사건의 배후에 조직적인 범죄가 있을 가능성을 제기하며, 피해자들에게는 즉각적인 계좌 점검과 보안 강화 조치를 권고하고 있습니다. 금융당국은 이번 사건의 원인을 철저히 조사하고 있으며, 향후 유사 사건을 방지하기 위한 대책 마련에 나설 계획입니다 
---
검색된 문서 수: 3
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [23]:
reranker_klue = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank_klue(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker_klue.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates_klue = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(questions_klue[0])
top3_klue = rerank_klue(questions_klue[0], candidates_klue, top_k=3)
print(f"후보 {len(candidates_klue)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3_klue[0].page_content[:200])


후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [24]:
def advanced_rag_klue(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank_klue(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv_klue, ctx_adv_klue = advanced_rag_klue(questions_klue[0])
print("Advanced RAG(KLUE) 답변:\n", ans_adv_klue)


Advanced RAG(KLUE) 답변:
 200여 개의 계좌입니다.


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [25]:
# Naive RAG(KLUE) 로 답변 + 컨텍스트 수집
naive_answers_klue, naive_contexts_klue = [], []
for q in questions_klue:
    ctx = naive_retriever_klue.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers_klue.append(a)
    naive_contexts_klue.append([d.page_content for d in ctx])

# Advanced RAG(KLUE) 로 답변 + 컨텍스트 수집
adv_answers_klue, adv_contexts_klue = [], []
for q in questions_klue:
    a, ctx = advanced_rag_klue(q)
    adv_answers_klue.append(a)
    adv_contexts_klue.append([d.page_content for d in ctx])

def make_dataset_klue(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions_klue,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths_klue,
    })

naive_ds_klue = make_dataset_klue(naive_answers_klue, naive_contexts_klue)
adv_ds_klue   = make_dataset_klue(adv_answers_klue,   adv_contexts_klue)

print("=== Naive RAG (KLUE) 채점 ===")
naive_result_klue = evaluate(naive_ds_klue, metrics=metrics,
                              llm=judge_llm, embeddings=judge_emb,
                              raise_exceptions=False)

print("=== Advanced RAG (KLUE) 채점 ===")
adv_result_klue = evaluate(adv_ds_klue, metrics=metrics,
                            llm=judge_llm, embeddings=judge_emb,
                            raise_exceptions=False)

naive_df_klue = naive_result_klue.to_pandas()
adv_df_klue   = adv_result_klue.to_pandas()

compare_klue = pd.concat([summary(naive_df_klue, "Naive RAG (KLUE)"),
                          summary(adv_df_klue,   "Advanced RAG (KLUE)")], axis=1)
print(compare_klue.round(3))
print("\nDelta (Advanced - Naive):")
print((compare_klue["Advanced RAG (KLUE)"] - compare_klue["Naive RAG (KLUE)"]).round(3))

print("\n=== 참고: KorQuAD 결과 ===")
print(compare.round(3))


=== Naive RAG (KLUE) 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG (KLUE) 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

                   Naive RAG (KLUE)  Advanced RAG (KLUE)
faithfulness                  0.525                0.800
answer_relevancy              0.276                0.229
context_precision             0.492                0.700
context_recall                0.550                0.700

Delta (Advanced - Naive):
faithfulness         0.275
answer_relevancy    -0.047
context_precision    0.208
context_recall       0.150
dtype: float64

=== 참고: KorQuAD 결과 ===
                   Naive RAG  Advanced RAG
faithfulness           0.575         0.850
answer_relevancy       0.291         0.262
context_precision      0.692         0.850
context_recall         0.750         0.850


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [26]:
from scipy import stats

EVAL_N_BIG = 60  # 표본을 늘려 통계적 신뢰도 확보 (50~100 권장)
questions_klue_big = [ex["question"] for ex in klue_samples[:EVAL_N_BIG]]
ground_truths_klue_big = [ex["answers"]["text"][0] for ex in klue_samples[:EVAL_N_BIG]]

naive_answers_big, naive_contexts_big = [], []
for q in questions_klue_big:
    ctx = naive_retriever_klue.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers_big.append(a)
    naive_contexts_big.append([d.page_content for d in ctx])

adv_answers_big, adv_contexts_big = [], []
for q in questions_klue_big:
    a, ctx = advanced_rag_klue(q)
    adv_answers_big.append(a)
    adv_contexts_big.append([d.page_content for d in ctx])

naive_ds_big = Dataset.from_dict({
    "user_input": questions_klue_big, "response": naive_answers_big,
    "retrieved_contexts": naive_contexts_big, "reference": ground_truths_klue_big,
})
adv_ds_big = Dataset.from_dict({
    "user_input": questions_klue_big, "response": adv_answers_big,
    "retrieved_contexts": adv_contexts_big, "reference": ground_truths_klue_big,
})

naive_result_big = evaluate(naive_ds_big, metrics=metrics, llm=judge_llm,
                             embeddings=judge_emb, raise_exceptions=False)
adv_result_big   = evaluate(adv_ds_big, metrics=metrics, llm=judge_llm,
                             embeddings=judge_emb, raise_exceptions=False)

naive_df_big = naive_result_big.to_pandas()
adv_df_big   = adv_result_big.to_pandas()

for col in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    t_stat, p_val = stats.ttest_rel(naive_df_big[col], adv_df_big[col])
    verdict = "유의함 (p<0.05)" if p_val < 0.05 else "유의하지 않음"
    print(f"{col:20s}  t={t_stat:.3f}  p={p_val:.4f}  -> {verdict}")


Evaluating:   0%|          | 0/240 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/240 [00:00<?, ?it/s]

faithfulness          t=-0.565  p=0.5742  -> 유의하지 않음
answer_relevancy      t=-0.304  p=0.7625  -> 유의하지 않음
context_precision     t=-4.042  p=0.0002  -> 유의함 (p<0.05)
context_recall        t=-3.227  p=0.0020  -> 유의함 (p<0.05)


### 마지막 Quiz — 직접 답을 적어보세요

1. **도메인 비교**: KorQuAD(위키) 와 KLUE-MRC(뉴스) 결과에서 4지표 중 가장 크게 달라진 건 무엇이었나요? 뉴스 기사의 어떤 특성(시점 표현, 인용, 숫자 등) 때문이라고 보이나요?
2. **Advanced 효과**: KLUE-MRC 에서도 Naive → Advanced 개선폭이 컸나요? KorQuAD 와 같았나요, 달랐나요?
3. **`is_impossible` 케이스**: Step B 에서 답할 수 없는 질문을 의도적으로 섞어 평가하면 어떤 지표가 가장 망가질까요? (실험해 보면 더 좋음)
4. (선택) 같은 파이프라인을 **MIRACL ko** 로 옮기면 어떤 차이가 있을지 예상해 보세요.

**1. 도메인 비교**

KLUE-MRC(뉴스)는 KorQuAD(위키)보다 4개 지표 전부 절대값이 낮다. 그중 가장 크게 벌어진 건 `context_precision`이다 (Naive 기준 KorQuAD 대비 −0.200).

| 지표 | KorQuAD (Naive) | KLUE (Naive) | 차이 |
|---|---|---|---|
| context_precision | 0.692 | 0.492 | **−0.200** |
| context_recall | 0.750 | 0.550 | −0.200 |
| faithfulness | 0.575 | 0.525 | −0.050 |
| answer_relevancy | 0.291 | 0.276 | −0.015 |

뉴스 기사는 위키 문서보다 날짜·인용문·수치·유사 인물명이 많이 겹친다. 임베딩 유사도만으로는 "진짜 정답이 있는 기사"와 "비슷한 사건을 다룬 다른 기사"를 구분하기 어려워, 검색(retrieval) 단계에서부터 정답 문서를 못 찾는 비율이 늘어난 것으로 해석할 수 있다. 다만 이번 재실행에서는 격차(−0.200)가 이전 실행(−0.250)보다 다소 줄어, 판단이 애매한 경계 사례의 영향도 일부 있는 것으로 보인다.

**2. Advanced 효과 비교**

| Delta (Advanced − Naive) | KorQuAD | KLUE | 표본 60개 t-test (p-value) |
|---|---|---|---|
| faithfulness | +0.275 | +0.275 (동일) | p=0.5742 → **유의하지 않음** |
| answer_relevancy | −0.029 | −0.047 | p=0.7625 → **유의하지 않음** |
| context_precision | +0.158 | +0.208 | p=0.0002 → **유의함** |
| context_recall | +0.100 | +0.150 | p=0.0020 → **유의함** |

표본 20개로 봤을 때는 KorQuAD와 KLUE 양쪽 모두 4개 지표가 전부 개선된 것처럼 보였다. 하지만 표본을 60개로 늘려 paired t-test를 해보면, 실제로 통계적으로 유의한 개선은 **`context_precision`과 `context_recall` 두 가지뿐**이다. `faithfulness`와 `answer_relevancy`에서 관찰된 변화(+0.275, −0.05 전후)는 p-value가 0.57~0.76으로 매우 높아, 우연(표본 노이즈)일 가능성이 크다.

즉 이전에 "Advanced RAG가 두 도메인 모두에서 `faithfulness`를 크게 개선한다"고 본 것은 표본이 작아서 생긴 착시였다. **Reranker(Advanced RAG)의 검증된 진짜 효과는 검색 품질(`context_precision`, `context_recall`) 개선에 한정되고, 이 효과가 답변의 충실도(`faithfulness`)나 관련성(`answer_relevancy`) 개선으로까지 통계적으로 유의하게 이어진다고는 말할 수 없다.**

이는 해설 가이드의 3번 항목("`faithfulness`가 살짝 떨어질 수 있다")과도 결이 맞는 결론이다 — 검색이 좋아졌다고 해서 답변 충실도가 자동으로 따라 좋아지는 건 아니며, 두 성질은 표본이 충분할 때는 서로 독립적으로 움직일 수 있다.

In [28]:
# ============================================================
# 실험 3 : is_impossible 케이스에서 RAG 파이프라인이 어떻게 무너지는지 확인
# ============================================================

# ---------- (1) 답할 수 없는 질문만 필터링 ----------
ds_klue_impossible = ds_klue.filter(lambda x: x["is_impossible"])

N_IMPOSSIBLE = 20
impossible_samples = list(ds_klue_impossible.shuffle(seed=42).select(range(N_IMPOSSIBLE)))

questions_impossible = [ex["question"] for ex in impossible_samples]

print(f"is_impossible=True 질문 {len(questions_impossible)}개 준비 완료")
print("예시:", questions_impossible[0])


# ---------- (2) Naive / Advanced 파이프라인으로 답변 생성 ----------
naive_answers_imp, naive_contexts_imp = [], []
for q in questions_impossible:
    ctx = naive_retriever_klue.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers_imp.append(a)
    naive_contexts_imp.append([d.page_content for d in ctx])

adv_answers_imp, adv_contexts_imp = [], []
for q in questions_impossible:
    a, ctx = advanced_rag_klue(q)
    adv_answers_imp.append(a)
    adv_contexts_imp.append([d.page_content for d in ctx])

print("답변 생성 완료")


# ---------- (3) 간단한 '거부(refusal) 탐지' — 정답이 없으므로 참조 기반 지표 대신 사용 ----------
REFUSAL_KEYWORDS = [
    "알 수 없", "없습니다", "찾을 수 없", "답변할 수 없", "확인할 수 없",
    "언급되지 않", "포함되어 있지 않", "제공된 문서에", "주어진 정보로는",
]

def is_refusal(answer: str) -> bool:
    return any(kw in answer for kw in REFUSAL_KEYWORDS)

naive_refusal_rate = sum(is_refusal(a) for a in naive_answers_imp) / len(naive_answers_imp)
adv_refusal_rate   = sum(is_refusal(a) for a in adv_answers_imp)   / len(adv_answers_imp)

print(f"\nNaive RAG   — '모른다'고 답한 비율(거부율): {naive_refusal_rate:.2%}")
print(f"Advanced RAG — '모른다'고 답한 비율(거부율): {adv_refusal_rate:.2%}")
print("(거부율이 낮을수록 = 답할 수 없는 질문에 그럴듯하게 지어내서 답했다는 뜻 → 환각 위험)")


# ---------- (4) RAGAS 채점 — reference 가 없으므로 참조 불필요 지표만 사용 ----------
from ragas.metrics import faithfulness, answer_relevancy

metrics_no_ref = [faithfulness, answer_relevancy]

def make_dataset_no_ref(questions, answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
    })

naive_ds_imp = make_dataset_no_ref(questions_impossible, naive_answers_imp, naive_contexts_imp)
adv_ds_imp   = make_dataset_no_ref(questions_impossible, adv_answers_imp,   adv_contexts_imp)

print("\n=== Naive RAG (is_impossible) 채점 ===")
naive_result_imp = evaluate(naive_ds_imp, metrics=metrics_no_ref,
                             llm=judge_llm, embeddings=judge_emb,
                             raise_exceptions=False)

print("=== Advanced RAG (is_impossible) 채점 ===")
adv_result_imp = evaluate(adv_ds_imp, metrics=metrics_no_ref,
                           llm=judge_llm, embeddings=judge_emb,
                           raise_exceptions=False)

naive_df_imp = naive_result_imp.to_pandas()
adv_df_imp   = adv_result_imp.to_pandas()

# ↓↓↓ 여기가 수정된 부분: 기존 summary() 대신 실제 존재하는 컬럼만 평균 ----------
def summary_no_ref(df, label):
    cols = ["faithfulness", "answer_relevancy"]  # 이 실험에서 계산 가능한 지표만
    avg = df[cols].mean()
    avg.name = label
    return avg

compare_imp = pd.concat([
    summary_no_ref(naive_df_imp, "Naive RAG (impossible)"),
    summary_no_ref(adv_df_imp,   "Advanced RAG (impossible)"),
], axis=1)

print("\n=== 비교 (참조 없이 계산 가능한 지표만) ===")
print(compare_imp.round(3))
print("\nDelta (Advanced - Naive):")
print((compare_imp["Advanced RAG (impossible)"] - compare_imp["Naive RAG (impossible)"]).round(3))


# ---------- (5) 실제로 어떤 식으로 답했는지 몇 개 눈으로 확인 ----------
print("\n=== 샘플 케이스 확인 ===")
for i in range(3):
    print(f"\nQ: {questions_impossible[i]}")
    print(f"Naive 답변   : {naive_answers_imp[i]}")
    print(f"Advanced 답변: {adv_answers_imp[i]}")

is_impossible=True 질문 20개 준비 완료
예시: 웹 드라마가 끝난 달은?
답변 생성 완료

Naive RAG   — '모른다'고 답한 비율(거부율): 85.00%
Advanced RAG — '모른다'고 답한 비율(거부율): 65.00%
(거부율이 낮을수록 = 답할 수 없는 질문에 그럴듯하게 지어내서 답했다는 뜻 → 환각 위험)

=== Naive RAG (is_impossible) 채점 ===


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

=== Advanced RAG (is_impossible) 채점 ===


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]


=== 비교 (참조 없이 계산 가능한 지표만) ===
                  Naive RAG (impossible)  Advanced RAG (impossible)
faithfulness                       0.408                      0.450
answer_relevancy                   0.070                      0.136

Delta (Advanced - Naive):
faithfulness        0.042
answer_relevancy    0.066
dtype: float64

=== 샘플 케이스 확인 ===

Q: 웹 드라마가 끝난 달은?
Naive 답변   : 웹 드라마 '간서치열전'은 최근 종방하였습니다.
Advanced 답변: 웹 드라마 '간서치열전'은 최근 종방하였으며, 구체적인 종료 달은 문서에 명시되어 있지 않습니다.

Q: 매해 연구개발 인력을 축소시키고 있다고 한 사람은?
Naive 답변   : 이인재 한국노동연구원장입니다.
Advanced 답변: 이인재 한국노동연구원장입니다.

Q: 가계 재무건전성이 매우 위험하다고 분석한 기관은?
Naive 답변   : 문서에 해당 내용이 포함되어 있지 않습니다.
Advanced 답변: 문서에 해당 내용이 포함되어 있지 않습니다.


**3. `is_impossible` 케이스**

실험 결과, 답할 수 없는 질문(is_impossible=True) 20개에 대해:

| 지표 | Naive RAG | Advanced RAG | Delta |
|---|---|---|---|
| faithfulness | 0.408 | 0.450 | +0.042 |
| answer_relevancy | 0.070 | 0.136 | +0.066 |

가장 눈에 띄게 무너진 지표는 **`answer_relevancy`**다. 정상적으로 답할 수 있는 질문(KLUE 전체 기준 0.2~0.28대)과 비교하면 0.070~0.136으로 절반 이하 수준까지 떨어졌다. 답할 수 없는 질문에는 "모른다"는 취지로 답하거나 애매하게 얼버무리는 경우가 많아, 질문을 역추론했을 때 원래 질문과의 유사도 자체가 낮게 나오기 때문으로 해석된다.

`faithfulness` 역시 정상 케이스(0.42~0.55대)보다 낮은 0.408~0.450으로, 근거 없는 내용을 만들어내는 경향이 어느 정도 반영됐다.

한편 **`context_precision`, `context_recall`은 애초에 계산이 불가능하다.** 이 지표들은 정답(reference)이 있어야 계산되는데, is_impossible 질문은 정의상 정답이 없기 때문이다. 이건 오히려 중요한 시사점이다 — **retrieval 품질을 정량적으로 감시하는 지표 자체가, 가장 감시가 필요한 "답할 수 없는 질문" 상황에서는 작동하지 않는다.** 이 공백은 RAGAS의 참조 기반 지표만으로는 메울 수 없고, 실제 서비스에서는 "얼마나 자신 있게 틀린 답을 하는가"를 감지하는 별도 장치(거부 여부 판단, confidence calibration 등)가 필요하다는 뜻이다.

실제 샘플을 보면 이 문제가 뚜렷하다:
- 정답이 아예 없는 "매해 연구개발 인력을 축소시키고 있다고 한 사람은?"이라는 질문에 Naive와 Advanced 둘 다 "이인재 한국노동연구원장입니다"라고 **똑같이 확신에 찬 오답**을 냈다 — Reranker가 검색 품질을 높였다고 해서 환각을 막아주는 건 아니라는 뜻이다.
- 반면 "웹 드라마가 끝난 날은?" 질문에서는 Advanced RAG만 "구체적인 종료 날짜는 문서에 명시되어 있지 않습니다"라고 부분적으로 한계를 인정했다 — 정제된 컨텍스트가 있을 때는 모델이 불확실성을 더 잘 드러내기도 한다는 뜻이다.
- "가계 재무건전성이 매우 위험하다고 분석한 기관은?" 같은 경우는 둘 다 정확히 거부했다.

**결론**: 정량 지표 중 가장 크게 붕괴하는 건 `answer_relevancy`이지만, 더 근본적인 문제는 **retrieval 품질 지표 자체가 무력화된다는 것**과, **Advanced RAG가 검색은 개선해도 환각(hallucination)을 막아주는 보장은 없다는 것**이다 (위 두 번째 샘플처럼 둘 다 틀린 답을 확신 있게 낼 수 있다).

**4. MIRACL ko 예상 (실행 전 예측)**

- **`context_precision`/`context_recall`은 KLUE보다 더 낮아질 가능성**: MIRACL은 후보 문서 풀이 훨씬 크고 노이즈가 많아, Naive RAG의 top-3 검색이 정답 문서를 놓칠 확률이 높아진다.
- **Advanced RAG(Reranker)의 상대적 개선폭은 오히려 더 커질 가능성**: 후보 풀이 클수록 k=10 → rerank 전략의 가치가 커지기 때문이다.
- **`faithfulness`/`answer_relevancy` 개선은 이번 실험(t-test)처럼 유의하지 않을 가능성**: 지금까지 결과상 이 두 지표는 표본을 늘리면 유의성이 사라지는 패턴을 보였으므로, MIRACL에서도 같은 패턴이 반복될 것으로 예상된다.

다만 이는 실측 데이터 없는 추측이며, 확인하려면 MIRACL ko 로드 → 동일 파이프라인 적용 → RAGAS 채점(표본 60개 이상) 실험이 필요하다.

## 마치며

이번 실습에서는 한국어 QA 벤치마크 위에서 다음을 진행했습니다.

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF)** / HyDE / Cross-encoder Reranking 적용
- ‘넓게 검색 → Reranker 로 좁힘 → LLM 답변’ Advanced RAG 체인 조립
- **Self-RAG** 패턴 — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive vs Advanced 를 정량 비교
- 추가 실습으로 도메인을 옮긴 **KLUE-MRC (뉴스 기반 한국어 MRC)** 에서 같은 파이프라인 재구성

**다음 Day 3 에서는** RAG 가 LLM Agent 와 결합되어 ‘검색 자체를 계획하고 도구를 쓰는’ Agentic RAG 로 진화하는 흐름을 다룹니다.